In [1]:
import os
from multiprocess import set_start_method
set_start_method("spawn")
import logging
logger = logging.getLogger("ignite.handlers.early_stopping.EarlyStopping")
logger.setLevel(logging.WARNING)

In [2]:
# import wandb 
from coreset_trainer.custom_olmo import DecomposedOlmo2
# from coreset_trainer.custom_phi import DecomposedPhiCausalLM

from transformers import Trainer, AutoModelForCausalLM, AutoTokenizer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

from coreset_trainer.trainer import CoresetTrainer
import os
import torch

import argparse

from coreset_trainer.train import tokenize_dataset

In [3]:
from influence_estimation.data_inf import DataInfEstimator
from influence_estimation.less_inf import LESSEstimator
from datasets import load_dataset


# base_model_path = "distilbert/distilgpt2"
# base_model_path = "allenai/OLMo-2-0425-1B"
# base_model_path = "allenai/OLMo-2-0425-1B"
# adapter_path = "/srv/home/users/loriss21cs/cfe/models/distilgpt2_tulu-v2-sft-mixture"
# adapter_path = "/srv/home/users/loriss21cs/cfe/models/OLMo-2-0425-1B_tulu-v2-sft-mixture"

train_dataset = load_dataset("allenai/tulu-v2-sft-mixture", split="train").shuffle(seed=42).select(range(1000))
train_dataset = train_dataset.map(
    lambda example, idx: {"indices": idx},
    with_indices=True
)





test_dataset = load_dataset("allenai/tulu-v2-sft-mixture", split="train").shuffle(seed=0).select(range(1000))
test_dataset = test_dataset.map(
    lambda example, idx: {"indices": idx},
    with_indices=True
)

In [4]:
estimators = [
    # LESSEstimator(base_model_path, adapter_path, train_dataset, test_dataset,fast_implementation=False),
    # LESSEstimator(base_model_path, adapter_path, train_dataset, test_dataset,fast_implementation=True),
    LESSEstimator("./models/pythia-31m_tulu-v2-sft-mixture_train", train_dataset, test_dataset),
    LESSEstimator("./models/pythia-31m_tulu-v2-sft-mixture_train", train_dataset, test_dataset, normalize=False),
    DataInfEstimator("./models/pythia-31m_tulu-v2-sft-mixture_train",train_dataset, test_dataset)
   
]

In [40]:
import pandas as pd
from scipy.stats import spearmanr

def rowwise_rank_corr(df1: pd.DataFrame, df2: pd.DataFrame):    
    corrs = []
    for i in range(df1.shape[0]):
        corr = spearmanr(df1.iloc[i], df2.iloc[i]).correlation
        corrs.append(corr)
    return pd.Series(corrs, index=df1.index)

corrs_01 = rowwise_rank_corr(estimators[0].influence_estimate,
                             estimators[1].influence_estimate)
corrs_02 = rowwise_rank_corr(estimators[0].influence_estimate,
                             estimators[2].influence_estimate)
corrs_12 = rowwise_rank_corr(estimators[1].influence_estimate,
                             estimators[2].influence_estimate)


agreement = pd.DataFrame({
    "LESS_norm_vs_unorm": corrs_01,
    "LESS_norm_vs_DataInf": corrs_02,
    "LESS_unorm_vs_DataInf": corrs_12,
})

print(agreement.describe())


       LESS_norm_vs_unorm  LESS_norm_vs_DataInf  LESS_unorm_vs_DataInf
count         1000.000000           1000.000000            1000.000000
mean             0.909813              0.333481               0.513240
std              0.040334              0.156783               0.171884
min              0.734755             -0.158971              -0.106454
25%              0.893000              0.232858               0.414164
50%              0.912363              0.354583               0.555562
75%              0.930225              0.450131               0.637392
max              0.999620              0.661027               0.807110


In [5]:
# parser = argparse.ArgumentParser()
# parser.add_argument("--ratio", type=float, default=0.5, help="Ratio parameter for coreset")
# args = parser.parse_args([])



# base_model_path = "gpt2"
# train_dataset_path = "allenai/tulu-v2-sft-mixture"


# ft_model_name = os.path.basename(base_model_path) + "_" + os.path.basename(train_dataset_path) + "_" + str(args.ratio).replace(".","p")


# os.environ["WANDB_PROJECT"] = "cfe_finetuning"
# os.environ["WANDB_NAME"] = ft_model_name





# tokenizer = AutoTokenizer.from_pretrained(base_model_path)
# tokenizer.chat_template = "{{ bos_token }}{% for message in messages %}{% if message['role'] == 'system' %}{{ '<|system|>\n' + message['content'] + '\n' }}{% elif message['role'] == 'user' %}{{ '<|user|>\n' + message['content'] + '\n' }}{% elif message['role'] == 'assistant' %}{% if not loop.last %}{{ '<|assistant|>\n'  + message['content'] + eos_token + '\n' }}{% else %}{{ '<|assistant|>\n'  + message['content'] + eos_token }}{% endif %}{% endif %}{% if loop.last and add_generation_prompt %}{{ '<|assistant|>\n' }}{% endif %}{% endfor %}" # https://huggingface.co/allenai/OLMo-2-1124-7B-SFT/blob/main/tokenizer_config.json 
# lora_config = LoraConfig(
#     r=4,
#     lora_alpha=32,
#     target_modules=["c_attn", "q_proj", "v_proj"],
#     lora_dropout=0.1,
#     bias="none",
#     task_type=TaskType.CAUSAL_LM
# )
# model = AutoModelForCausalLM.from_pretrained(base_model_path)

# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token
# model.resize_token_embeddings(len(tokenizer))



# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()

# if hasattr(model, "enable_input_require_grads"):
#         model.enable_input_require_grads()
# else:
#         def make_inputs_require_grad(module, input, output):
#             output.requires_grad_(True)
#         model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)


# # if training_args.data_selection_unit == "mezo" and training_args.efficient_mezo:
# # model.decomposer = DecomposedOlmo2(model)
# train_dataset = load_dataset(train_dataset_path, split="train[0:10]")

# def add_index(example, idx):
#     example["indices"] = idx
#     return example



# print(train_dataset)


# # tokenized_train = tokenize_dataset(train_dataset, tokenizer, max_lenght=1024)
# # tokenized_train = tokenized_train.map(add_index, with_indices=True)
# # data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
# torch.cuda.empty_cache()



In [6]:
%%writefile validation_engine.py
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
import numpy as np
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling
from torch.cuda.amp import autocast
from tqdm import tqdm
import copy
from transformers import DataCollatorForLanguageModeling

class ChatTemplateCollator:
    def __init__(self, tokenizer, mlm=False, max_length=1024): # TODO
        self.tokenizer = tokenizer
        
        self.max_length = max_length
        self.base_collator = DataCollatorForLanguageModeling(
            tokenizer=tokenizer, mlm=mlm
        )

    def __call__(self, features):
        texts = [
            self.tokenizer.apply_chat_template(
                f["messages"], tokenize=False, add_generation_prompt=False
            )
            for f in features
        ]
        tokenized = self.tokenizer(
            texts,
            truncation=True,
            return_tensors="pt",
            padding=True,
            max_length=self.max_length
        )
 
        batch = [
            {"input_ids": tokenized["input_ids"][i],
             "attention_mask": tokenized["attention_mask"][i]}
            for i in range(len(tokenized["input_ids"]))
        ]
        return self.base_collator(batch)

class ValidationEngine():
    def __init__(self, base_model_path, device):
        self.base_model_path = base_model_path
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(self.base_model_path, padding_side='left')
        self.data_collator = ChatTemplateCollator(tokenizer=self.tokenizer, mlm=False)
        self.model_ = None
        if self.tokenizer.chat_template is None:
            self.tokenizer.chat_template = "{{ bos_token }}{% for message in messages %}{% if message['role'] == 'system' %}{{ '<|system|>\n' + message['content'] + '\n' }}{% elif message['role'] == 'user' %}{{ '<|user|>\n' + message['content'] + '\n' }}{% elif message['role'] == 'assistant' %}{% if not loop.last %}{{ '<|assistant|>\n'  + message['content'] + eos_token + '\n' }}{% else %}{{ '<|assistant|>\n'  + message['content'] + eos_token }}{% endif %}{% endif %}{% if loop.last and add_generation_prompt %}{{ '<|assistant|>\n' }}{% endif %}{% endfor %}" # https://huggingface.co/allenai/OLMo-2-1124-7B-SFT/blob/main/tokenizer_config.json 
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token # TODO
    def get_log_p(self, model, examples):
        batch = self.data_collator(examples)
        batch = {k: v.to(self.device) for k, v in batch.items()}
        with torch.no_grad(): 
            outputs = model.generate(**batch, max_new_tokens=5, return_dict_in_generate=True, output_scores=True,  num_beams=1)
            transition_scores = model.compute_transition_scores(
                outputs.sequences, outputs.scores, normalize_logits=True
            )
            log_p = transition_scores.sum(axis=1)
            # print("log_p",log_p)
            return log_p
    @property
    def model(self):
        if self.model_ is None:
            self.model_ = AutoModelForCausalLM.from_pretrained(self.base_model_path,torch_dtype=torch.float32).to(self.device)
            # self.model_.generation_config.use_cache = True
        return self.model_
    def score(self, train_dataset, test_sets):
        deltas = []
        model_ft = AutoModelForCausalLM.from_pretrained(self.base_model_path, torch_dtype=torch.float32).to(self.device)
        model_ft = self.finetune(model_ft, train_dataset)
        for test_dataset in test_sets:
            log_p_before_ft = self.get_log_p(self.model, test_dataset)       
            log_p_after_ft = self.get_log_p(model_ft, test_dataset)
            deltas.append(log_p_after_ft - log_p_before_ft)
        return deltas
    def finetune(self, model, train_dataset):
        training_args = TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        remove_unused_columns=False,
        num_train_epochs=5,
        learning_rate=2e-5,
        logging_steps=500,
        seed=42,
        #save_steps=None,#100,
        # save_total_limit=2,
        #  fp16=True, 
        #   bf16=True,
        report_to=[],
        disable_tqdm=True 
        )
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            processing_class=self.tokenizer,
            data_collator=self.data_collator,
        )
        

        trainer.train(resume_from_checkpoint=False)
        return model

Overwriting validation_engine.py


In [7]:
%%writefile validation_worker.py
import os
import torch
from tqdm import tqdm
CACHE_DIR = "./cache/validation/"


def process(batch, gpu_id):
    """
    Worker function to process a batch of test sets on a specific GPU.
    """
    os.makedirs(CACHE_DIR,exist_ok=True)
    try:
        # Restrict this process to a single GPU
        os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
        import torch
        from validation_engine import ValidationEngine  # import inside worker

        engine = ValidationEngine("EleutherAI/pythia-31m", device="cuda:0")
        results = []
        for ts_batch, test_sets, test_indices, estimator, explanation_type in tqdm(batch, desc=f"GPU {gpu_id}", leave=False,position=gpu_id):
          
            deltas =  [engine.score(explanation, test_sets) for (setting, explanation) in ts_batch]
            results.append([(setting,
                             estimator.get_config_string().replace(" ",""), # estimator
                             explanation,
                             explanation_type, # type
                             test_indices, 
                             delta[0].item(),
                             ) for (setting, explanation), delta in zip(ts_batch, deltas)])
        return results

    except Exception as e:
        import traceback, logging
        logging.error(f"Error in process for GPU {gpu_id}: {e}")
        logging.error(traceback.format_exc())
        raise


Overwriting validation_worker.py


In [8]:
import logging
from concurrent.futures import ProcessPoolExecutor, as_completed
import torch
import multiprocessing
from explanations import TopKMostInfluential, TopKLeastInfluential, TopKMostOrthogonal, TopKLeastOrthogonal
from tqdm import tqdm
import itertools
import pandas as pd
import traceback

import evaluation_worker

logging.basicConfig(level=logging.ERROR, format='%(asctime)s [%(levelname)s] %(message)s')
multiprocessing.set_start_method('spawn', force=True)
torch.manual_seed(42)


n_test = 10
test_indices = test_dataset.shuffle(seed=42).select(range(n_test))["indices"]

num_devices = torch.cuda.device_count()

explanation_types = [TopKMostInfluential, TopKLeastInfluential, TopKMostOrthogonal, TopKLeastOrthogonal]






In [9]:
import logging
from concurrent.futures import ProcessPoolExecutor, as_completed
import torch
import multiprocessing
from validation_worker import process

logging.basicConfig(level=logging.ERROR, format='%(asctime)s [%(levelname)s] %(message)s')

multiprocessing.set_start_method('spawn', force=True)

num_gpus = torch.cuda.device_count()
logging.info(f"Detected {num_gpus} GPUs")

[INFO] Detected 2 GPUs


In [10]:
results = []
test_sets = []
for estimator in estimators:
    for explanation_type in explanation_types:
        explanations = estimator.influence_estimate.iloc[test_indices].apply(
                lambda row: explanation_type(row.name, estimator), axis=1
            )
        documents = [e.documents for e in explanations]
        test_indices = [e.dataset_idx for e in explanations]
        test_sets.extend([(
            ((("explanation", train_dataset.select(e))),
            ("random", train_dataset.shuffle(seed=seed).select(range(len(e)))),
            ),
            [test_dataset.select([idx])], [[idx]], estimator, explanation_type) for seed, (e,idx) in enumerate(zip(documents,test_indices))
        ])

batch_size = max(len(test_sets) // num_gpus, 1)
batches = [test_sets[i:i + batch_size] for i in range(0, len(test_sets), batch_size)]
      

with ProcessPoolExecutor(max_workers=num_gpus) as executor:
    futures = [executor.submit(process, batch, i % num_gpus) for i, batch in enumerate(batches)]
    
    for future in as_completed(futures):
        try:
            results.extend(future.result())
        except Exception as e:
            logging.error(f"A future failed: {e}")


#     break
# break

logging.info("All batches processed")


GPU 0:   0%|          | 0/60 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
GPU 0:   2%|▏         | 1/60 [00:17<16:43, 17.01s/it]Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.

GPU 1:   2%|▏         | 1/60 [00:19<19:03, 19.38s/it]Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_

{'train_runtime': 6.6736, 'train_samples_per_second': 7.492, 'train_steps_per_second': 7.492, 'train_loss': 2.511186370849609, 'epoch': 5.0}
{'train_runtime': 6.2901, 'train_samples_per_second': 7.949, 'train_steps_per_second': 7.949, 'train_loss': 2.3392115783691407, 'epoch': 5.0}
{'train_runtime': 6.1442, 'train_samples_per_second': 8.138, 'train_steps_per_second': 8.138, 'train_loss': 2.5912884521484374, 'epoch': 5.0}
{'train_runtime': 5.8908, 'train_samples_per_second': 8.488, 'train_steps_per_second': 8.488, 'train_loss': 2.529496917724609, 'epoch': 5.0}
{'train_runtime': 4.9895, 'train_samples_per_second': 10.021, 'train_steps_per_second': 10.021, 'train_loss': 2.8594741821289062, 'epoch': 5.0}
{'train_runtime': 5.6432, 'train_samples_per_second': 8.86, 'train_steps_per_second': 8.86, 'train_loss': 2.4237994384765624, 'epoch': 5.0}
{'train_runtime': 5.6077, 'train_samples_per_second': 8.916, 'train_steps_per_second': 8.916, 'train_loss': 2.8812945556640623, 'epoch': 5.0}
{'train_

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


{'train_runtime': 8.9777, 'train_samples_per_second': 5.569, 'train_steps_per_second': 5.569, 'train_loss': 2.7346670532226565, 'epoch': 5.0}
{'train_runtime': 7.1623, 'train_samples_per_second': 6.981, 'train_steps_per_second': 6.981, 'train_loss': 2.3523313903808596, 'epoch': 5.0}
{'train_runtime': 6.8947, 'train_samples_per_second': 7.252, 'train_steps_per_second': 7.252, 'train_loss': 2.672760009765625, 'epoch': 5.0}
{'train_runtime': 5.6983, 'train_samples_per_second': 8.775, 'train_steps_per_second': 8.775, 'train_loss': 2.5161984252929686, 'epoch': 5.0}
{'train_runtime': 6.3871, 'train_samples_per_second': 7.828, 'train_steps_per_second': 7.828, 'train_loss': 2.6475540161132813, 'epoch': 5.0}
{'train_runtime': 5.3299, 'train_samples_per_second': 9.381, 'train_steps_per_second': 9.381, 'train_loss': 2.419933166503906, 'epoch': 5.0}
{'train_runtime': 5.6533, 'train_samples_per_second': 8.844, 'train_steps_per_second': 8.844, 'train_loss': 2.665640563964844, 'epoch': 5.0}
{'train_r

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
GPU 0:  50%|█████     | 30/60 [07:16<07:19, 14.64s/it]Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.

GPU 1:  50%|█████     | 30/60 [07:19<07:18, 14.63s/it]Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
GPU 0:  52%|█████▏    | 31/60 [07:29<06:53, 14.27s/it]Setting `pad_token_id

...

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


{'train_runtime': 5.5881, 'train_samples_per_second': 8.948, 'train_steps_per_second': 8.948, 'train_loss': 2.5308186340332033, 'epoch': 5.0}
{'train_runtime': 5.5116, 'train_samples_per_second': 9.072, 'train_steps_per_second': 9.072, 'train_loss': 2.806283264160156, 'epoch': 5.0}
{'train_runtime': 6.1541, 'train_samples_per_second': 8.125, 'train_steps_per_second': 8.125, 'train_loss': 2.5968612670898437, 'epoch': 5.0}
{'train_runtime': 6.1464, 'train_samples_per_second': 8.135, 'train_steps_per_second': 8.135, 'train_loss': 2.3421511840820313, 'epoch': 5.0}
{'train_runtime': 6.2721, 'train_samples_per_second': 7.972, 'train_steps_per_second': 7.972, 'train_loss': 2.8151095581054686, 'epoch': 5.0}
{'train_runtime': 6.2383, 'train_samples_per_second': 8.015, 'train_steps_per_second': 8.015, 'train_loss': 2.5277490234375, 'epoch': 5.0}
{'train_runtime': 6.5339, 'train_samples_per_second': 7.652, 'train_steps_per_second': 7.652, 'train_loss': 2.6659759521484374, 'epoch': 5.0}
{'train_ru

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
GPU 0:  98%|█████████▊| 59/60 [14:11<00:14, 14.34s/it]Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.

GPU 1:  98%|█████████▊| 59/60 [14:14<00:14, 14.31s/it]Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting

{'train_runtime': 5.7253, 'train_samples_per_second': 8.733, 'train_steps_per_second': 8.733, 'train_loss': 2.035836486816406, 'epoch': 5.0}
{'train_runtime': 5.6258, 'train_samples_per_second': 8.888, 'train_steps_per_second': 8.888, 'train_loss': 2.3514976501464844, 'epoch': 5.0}
{'train_runtime': 5.7831, 'train_samples_per_second': 8.646, 'train_steps_per_second': 8.646, 'train_loss': 2.5317152404785155, 'epoch': 5.0}
{'train_runtime': 5.6112, 'train_samples_per_second': 8.911, 'train_steps_per_second': 8.911, 'train_loss': 2.806283264160156, 'epoch': 5.0}
{'train_runtime': 6.2757, 'train_samples_per_second': 7.967, 'train_steps_per_second': 7.967, 'train_loss': 2.675413818359375, 'epoch': 5.0}
{'train_runtime': 6.2079, 'train_samples_per_second': 8.054, 'train_steps_per_second': 8.054, 'train_loss': 2.3514976501464844, 'epoch': 5.0}
{'train_runtime': 6.4646, 'train_samples_per_second': 7.734, 'train_steps_per_second': 7.734, 'train_loss': 2.6587396240234376, 'epoch': 5.0}
{'train_r

[INFO] All batches processed


In [44]:
results

[{'explanation_type': "<class 'explanations.TopKLeastInfluential'>",
  'estimator': 'DataInfEstimator:fast_implementation=True',
  't_stat': 0.8266051212848249,
  'p_val': 0.2149149383739613,
  'mean_diff': 0.9929018020629883},
 {'explanation_type': "<class 'explanations.TopKLeastInfluential'>",
  'estimator': 'LESSEstimator:normalize=False',
  't_stat': 2.311290985942926,
  'p_val': 0.023069453099132217,
  'mean_diff': 1.9455487966537475},
 {'explanation_type': "<class 'explanations.TopKLeastInfluential'>",
  'estimator': 'LESSEstimator:normalize=True',
  't_stat': 2.659904395866568,
  'p_val': 0.013024938475023406,
  'mean_diff': 2.081648755073547},
 {'explanation_type': "<class 'explanations.TopKLeastOrthogonal'>",
  'estimator': 'DataInfEstimator:fast_implementation=True',
  't_stat': 1.6205862766166925,
  'p_val': 0.06977879532349232,
  'mean_diff': 1.5247912168502809},
 {'explanation_type': "<class 'explanations.TopKLeastOrthogonal'>",
  'estimator': 'LESSEstimator:normalize=Fals

In [ ]:
import pandas as pd
df = pd.DataFrame([ x
    for xs in results
    for x in xs
],columns=["setting","estimator","explanation", "explanation_type", "test_instance_idx", "gain"])
df["test_instance_idx"] = df["test_instance_idx"].apply(lambda x: x[0][0])
df["explanation_type"] = df["explanation_type"].apply(str)

df.set_index(["setting", "estimator","explanation_type", "test_instance_idx"])


TypeError: 'float' object is not subscriptable

In [ ]:
# df = pd.concat([df,df,df])

In [ ]:
# group_random

In [37]:
import pandas as pd
from scipy.stats import ttest_rel

results = []

for (explanation_type, estimator), group in df.groupby(["explanation_type", "estimator"]):
    group_explanation = group[group["setting"] == "explanation"]["gain"]
    group_random = group[group["setting"] == "random"]["gain"]
    
    t_stat, p_val = ttest_rel(group_explanation, group_random, alternative="greater")
    mean_diff = group_explanation.mean() - group_random.mean()
    
    results.append({
        "explanation_type": explanation_type,
        "estimator": estimator,
        "t_stat": t_stat,
        "p_val": p_val,
        "mean_diff": mean_diff
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="p_val")


In [38]:
results_df[results_df["explanation_type"].str.contains("LeastIn")].sort_values(by="p_val")

,explanation_type,estimator,t_stat,p_val,mean_diff
2,<class 'explanations.TopKLeastInfluential'>,LESSEstimator:normalize=True,2.659904,0.013025,2.081649
1,<class 'explanations.TopKLeastInfluential'>,LESSEstimator:normalize=False,2.311291,0.023069,1.945549
0,<class 'explanations.TopKLeastInfluential'>,DataInfEstimator:fast_implementation=True,0.826605,0.214915,0.992902


In [39]:
results_df[results_df["explanation_type"].str.contains("MostIn")].sort_values(by="p_val")

,explanation_type,estimator,t_stat,p_val,mean_diff
6,<class 'explanations.TopKMostInfluential'>,DataInfEstimator:fast_implementation=True,1.554648,0.077226,1.470047
7,<class 'explanations.TopKMostInfluential'>,LESSEstimator:normalize=False,0.619080,0.275605,0.801013
8,<class 'explanations.TopKMostInfluential'>,LESSEstimator:normalize=True,-0.175070,0.567550,-0.307314


In [15]:
df.describe()

,test_instance_idx,gain
count,240.000000,240.000000
mean,649.400000,-0.297139
std,324.798396,2.375006
min,127.000000,-8.421473
25%,260.000000,-0.527687
50%,833.000000,0.509406
75%,916.000000,1.060665
max,978.000000,2.912562
